# Scraping World Population Data

This small project retrieves the World Population data table from the `List of countries and dependencies by population` Wikipedia article (as of March 31st 2026). 

To do this, the libraries `requests` and `BeautifulSoup` can be leveraged:

* A HTTP GET request is sent to the article URL and the corresponding HTML data is parsed using BeautifulSoup with the lxml parser.

* The resulting data is converted to a pandas dataframe and saved locally as a csv file.

### A Note to Readers
It is important to realise that scraping has grown a lot more complicated than simple HTML parsing. Often, it is highly inefficient or impossible to scrape specific data from modern websites by simply parsing HTML in the way that this project showcases. 

Modern E-commerce websites for instance often store data on the back-end rather than encoding it within HTML tags, with a heavy reliance on JavaScript. The user client responsible for displaying the front-end will make HTTP requests to the company's back-end server to request the data required to display the relevant data/information to the user. This process where the back-end provides the necessary data to the front-end for display is known as **hydration**. In these instances, we would require a different technique, such as attempting to reverse-engineer the API that is responsible for the hydration process.

Also, many modern websites employ more stringent anti-scraping techniques when compared to older websites. It may be necessary in some cases to change the method in which a request is sent to the URL to prevent the script being flagged and blocked. This can be achieved for example using libraries such as `curl_cffi` which is better at mimicking a web browser by **imitating a more consistent TLS fingerprint**.

## Scraping Script

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import lxml
from requests.exceptions import RequestException, HTTPError
from datetime import datetime

pd.options.display.max_rows = None

try:
    # send GET request to the specified URL with a custom User-Agent header to mimic a request from a web browser. This can help avoid being blocked by the website's anti-scraping measures.
    response = requests.get("https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population", headers={'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/144.0.0.0 Safari/537.36 OPR/128.0.0.0 (Edition std-2)"}) 
    
    response.raise_for_status() # check for successful req. if unsuccessful, a HTTPError is raised and handled below

except RequestException as e:
    print(f"An error occurred with your request: {e}")
    
except HTTPError as e:
    print(f"A HTTP error occurred: {e}")
    
# create instance of BeautifulSoup to parse the HTML content using the lxml parser. 
soup = BeautifulSoup(response.text, "lxml") 

# find the first table element with the class attribute "wikitable" - this corresponds to the table containing the population data on the Wikipedia page.
table = soup.find("table", class_="wikitable") 

td_elements = table.select("tr td") # select all td elements that are descendants of tr elements within the table. This will give us a list of all the table cells in the population data table, which we can then iterate through to extract the relevant information for each country.

td_elements[:12]

[<td><b><span class="flagicon" style="padding-left:25px;"> </span>World</b>
 </td>,
 <td>8,232,000,000</td>,
 <td><div class="center">100%</div></td>,
 <td><span data-sort-value="000000002025-06-13-0000" style="white-space:nowrap">13 Jun 2025</span>
 </td>,
 <td>UN projection<sup class="reference" id="cite_ref-UNFPA_1-1"><a href="#cite_note-UNFPA-1"><span class="cite-bracket">[</span>1<span class="cite-bracket">]</span></a></sup><sup class="reference" id="cite_ref-auto1_4-0"><a href="#cite_note-auto1-4"><span class="cite-bracket">[</span>3<span class="cite-bracket">]</span></a></sup></td>,
 <td>
 </td>,
 <td><span class="flagicon"><span class="mw-image-border" typeof="mw:File"><a href="/wiki/India" title="India"><img alt="India" class="mw-file-element" data-file-height="600" data-file-width="900" decoding="async" height="15" src="//upload.wikimedia.org/wikipedia/en/thumb/4/41/Flag_of_India.svg/40px-Flag_of_India.svg.png" srcset="//upload.wikimedia.org/wikipedia/en/thumb/4/41/Flag_of_In

Checking the first two rows, we see the first row contains the combined world population data. 

We can extract this separately.

### Extracting World Population Data

In [46]:
world_total = []

world_row = td_elements[:5]

for cell in world_row:
    world_total.append(cell.text.strip())

world_total[4] = soup.find("li", id="cite_note-UNFPA-1").find("a", class_="external text")["href"] # replace the reference text contained within the td element with the reference URL

world_total

for i in range(len(world_total)):
  
  if i == 1:
    world_total[i] = int(world_total[i].replace(",", ""))
    
  elif i == 3:
    world_total[i] = datetime.strptime(world_total[i], "%d %b %Y").date().strftime("%B %d %Y")

world_total

['World',
 8232000000,
 '100%',
 'June 13 2025',
 'https://www.unfpa.org/data/world-population-dashboard']

In [47]:
world_total.remove("World") # remove unnecessary data
world_total.remove("100%")

In [48]:
print(f"""
As of {world_total[1]}, there are approximately {world_total[0]:,} people on Earth.
      
Source: {world_total[-1]}
""")


As of June 13 2025, there are approximately 8,232,000,000 people on Earth.

Source: https://www.unfpa.org/data/world-population-dashboard



### Parsing Population by Country

In [49]:
country_rows = td_elements[6:] # skip the first 6 td elements, which correspond to the "World" row
data = []
all_dates_converted = True

for i in range(len(country_rows)):
    
    record = []
    
    if i % 6 == 0:
        country = country_rows[i].text.strip()
    
    if i % 6 == 1:
        population = int(country_rows[i].text.replace(",",""))
        
    if i % 6 == 2:
        percentage = float(country_rows[i].text.replace("%",""))
        
    if i % 6 == 3:
        try:
            date = datetime.strptime(country_rows[i].text.strip(), "%d %b %Y").date().strftime("%d %B %Y")
            
        except ValueError:
            print(f"Could not convert {country}\'s date data: '{country_rows[i].text.strip()}'")
            all_dates_converted = False
            date = country_rows[i].text.strip() # keep original text if conversion fails
    else: 
        continue
      
    record.append(country)
    record.append(population)
    record.append(percentage)
    record.append(date)
    
    data.append(record)

if all_dates_converted:
    print("")
    print("All dates were successfully converted to the desired format.")
    
columns = ["Country", "Population", "% of World", "Date"]

df = pd.DataFrame(data, columns=columns)
df

Could not convert Lesotho's date data: '2024'
Could not convert Comoros's date data: '2026'
Could not convert Bhutan's date data: '2025'
Could not convert São Tomé and Príncipe's date data: '2024'
Could not convert Grenada's date data: '2021'
Could not convert Dominica's date data: '2018'
Could not convert Saint Kitts and Nevis's date data: '2022'


,Country,Population,% of World,Date
0,India,1417492000,17.2000,01 July 2025
1,China,1404890000,17.1000,31 December 2025
2,United States,341784857,4.1000,01 July 2025
3,Indonesia,288315089,3.5000,31 December 2025
4,Pakistan,241499431,2.9000,01 March 2023
5,Nigeria,223800000,2.7000,01 July 2023
6,Brazil,213421037,2.6000,01 July 2025
7,Bangladesh,169828911,2.1000,14 June 2022
8,Russia,146028325,1.8000,01 January 2025
9,Mexico,131001723,1.6000,31 December 2025


In [50]:
df.to_csv("world_population_wikipedia.csv", index=False)